# 12. Conclusiones, limitaciones y checklist

Este capítulo **lee las tablas generadas en los capítulos anteriores** y resume los
hallazgos; las cifras salen de esas tablas, no están escritas a mano.

In [1]:
from pathlib import Path
import re, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

project_root = Path.cwd().resolve()
while not (project_root / "_config.yml").exists():
    if project_root.parent == project_root:
        raise FileNotFoundError("No se encontró _config.yml. Ejecuta el notebook dentro de la carpeta del libro.")
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

for _folder in ("tables", "figures"):          # se crean si no existen
    (project_root / _folder).mkdir(exist_ok=True)

from src.project_utils import *

configure_plots()
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 70)
TABLES = project_root / "tables"

def md_out(text):
    display(Markdown(text))

import platform, json
part = pd.read_csv(TABLES / "particion_pedidos.csv")
res = pd.read_csv(TABLES / "resultados_prueba.csv", index_col=0)
ic = pd.read_csv(TABLES / "intervalos_bootstrap_logistica.csv", index_col=0)
dif = pd.read_csv(TABLES / "diferencias_vs_dummy.csv", index_col=0)
cv = pd.read_csv(TABLES / "cv_resumen.csv")
con_sin = pd.read_csv(TABLES / "con_fuga_vs_sin_fuga.csv")
techo = pd.read_csv(TABLES / "techo_sin_fuga.csv").iloc[0]
sel = json.loads((TABLES / "modelo_elegido.json").read_text(encoding="utf-8"))
dec = pd.read_csv(TABLES / "decision_poblacion.csv").iloc[0]
rank = pd.read_csv(TABLES / "ranking_univariado.csv")
moran = pd.read_csv(TABLES / "moran_global.csv")
sub = pd.read_csv(TABLES / "desempeno_subgrupos.csv")

## Resumen ejecutivo

In [2]:
n_tr, n_te = (part["conjunto"] == "train").sum(), (part["conjunto"] == "test").sum()
lr_, du_ = res.loc["Regresión logística"], res.loc["Dummy (prior)"]
cv_lr = cv[(cv["esquema"].str.startswith("Temporal")) & (cv["modelo"] == "Regresión logística")].iloc[0]
md_out(
    f"Se construyó una línea base para predecir, **al crear un pedido**, si su entrega llegará tarde en **DataCo Smart Supply Chain** "
    f"(180.519 líneas, 2015–2018). La partición es **cronológica** ({es(n_tr, 0)} líneas de entrenamiento y {es(n_te, 0)} de prueba, con 7 días de separación) "
    f"y se excluyeron {es(int(dec['n_excluidos_train']), 0)} líneas de pedidos cancelados o sospechosos de fraude del entrenamiento (y sus equivalentes en la prueba).\n\n"
    f"- **Métrica principal, ROC-AUC (prueba):** regresión logística **{es(lr_['ROC-AUC'], 4)}** "
    f"(IC 95 % bootstrap por pedido: {es(ic.loc['ROC-AUC'].iloc[1], 4)} – {es(ic.loc['ROC-AUC'].iloc[2], 4)}) frente a 0,5 del Dummy. "
    f"En validación cruzada temporal: {cv_lr['roc_auc'].replace('.', ',')}.\n"
    f"- **Accuracy:** {es(lr_['Accuracy'], 3)} frente a {es(du_['Accuracy'], 3)} del Dummy (diferencia {es(dif.loc['Accuracy', 'Diferencia'], 3)}, "
    f"IC {es(dif.loc['Accuracy', 'IC inferior'], 3)} – {es(dif.loc['Accuracy', 'IC superior'], 3)}).\n"
    f"- **Techo sin fuga:** ROC-AUC {es(techo['ROC-AUC'], 3)} (tabla de tasas por modo × jornada). El modelo lo alcanza.\n"
    f"- **Con fuga** (días reales de envío) el ROC-AUC sería {es(con_sin.iloc[0, 1], 3)}: ese es el resultado «espectacular» que se descarta.\n"
    f"- **Variables del modelo:** {', '.join(sel['cat'] + sel['num_lin'] + sel['num_log'])} (modo de envío × jornada AM/PM del pedido).")

Se construyó una línea base para predecir, **al crear un pedido**, si su entrega llegará tarde en **DataCo Smart Supply Chain** (180.519 líneas, 2015–2018). La partición es **cronológica** (143.075 líneas de entrenamiento y 36.222 de prueba, con 7 días de separación) y se excluyeron 6.032 líneas de pedidos cancelados o sospechosos de fraude del entrenamiento (y sus equivalentes en la prueba).

- **Métrica principal, ROC-AUC (prueba):** regresión logística **0,7746** (IC 95 % bootstrap por pedido: 0,7668 – 0,7825) frente a 0,5 del Dummy. En validación cruzada temporal: 0,7743 ± 0,0035.
- **Accuracy:** 0,725 frente a 0,573 del Dummy (diferencia 0,152, IC 0,137 – 0,166).
- **Techo sin fuga:** ROC-AUC 0,775 (tabla de tasas por modo × jornada). El modelo lo alcanza.
- **Con fuga** (días reales de envío) el ROC-AUC sería 0,975: ese es el resultado «espectacular» que se descarta.
- **Variables del modelo:** mode_x_daypart (modo de envío × jornada AM/PM del pedido).

## Métrica elegida: ROC-AUC y por qué

La métrica principal del proyecto es el **área bajo la curva ROC (ROC-AUC)**. La elección se
justifica con el problema y con los propios resultados del libro:

1. **Coincide con el uso del modelo.** El objetivo operativo es **ordenar** los pedidos por
   riesgo de retraso para priorizar acciones (reasignar transporte, avisar al cliente). La
   ROC-AUC mide exactamente eso: la probabilidad de que un pedido tardío elegido al azar reciba
   un puntaje mayor que uno no tardío.
2. **No depende del umbral.** El corte 0,5 es arbitrario y lo decide la operación según su
   capacidad. Las métricas basadas en el umbral (accuracy, recall, F1) cambian mucho al moverlo;
   la ROC-AUC no (tabla de abajo).
3. **Las clases están casi balanceadas** (≈ 57 % tardías). Con este balance la ROC-AUC no se
   infla por la clase mayoritaria, y no hace falta una métrica para clases raras como la
   Average Precision, que se reporta solo como complemento.
4. **Da referencias claras.** El Dummy vale 0,5; el techo sin fuga, ≈ 0,775; con fuga se llega a
   ≈ 0,975. Así la misma escala sirve para medir la mejora sobre la línea base y para **detectar
   fuga**.
5. **Accuracy es engañosa como principal.** El Dummy ya acierta el 57 % sin aprender nada, y el
   valor depende del umbral y de la prevalencia del periodo.

Se descartó **precision** como métrica de reporte. Depende del umbral y, como la clase positiva
es la mayoritaria, cualquier regla trivial ya tiene una precision cercana a la prevalencia, así
que no informa sobre la capacidad de ordenar. Recall y F1 se mantienen solo para mostrar el
efecto operativo del umbral elegido, y el **Brier** junto con la curva de calibración verifican
que las probabilidades sean creíbles.

In [3]:
umb = pd.read_csv(TABLES / "umbral_alternativo.csv", index_col=0)
prev = pd.read_csv(TABLES / "objetivo_distribucion.csv", index_col=0)
sens = umb[["Accuracy", "Recall", "F1", "ROC-AUC"]].copy()
sens.loc["Dummy (prior)"] = res.loc["Dummy (prior)", ["Accuracy", "Recall", "F1", "ROC-AUC"]]
display(sens.round(4))
cambio = (umb.iloc[1] - umb.iloc[0])
md_out(f"Al pasar del umbral 0,5 al de Youden, accuracy cambia {es(cambio['Accuracy'], 3)}, recall {es(cambio['Recall'], 3)} y F1 {es(cambio['F1'], 3)}, "
       f"mientras que el ROC-AUC cambia **{es(cambio['ROC-AUC'], 3)}**. El Dummy obtiene F1 = {es(res.loc['Dummy (prior)', 'F1'], 3)} "
       f"(mayor que la regresión logística) solo por predecir siempre «tardía»: otra razón para no usar F1 como métrica principal. "
       f"Con la ROC-AUC, el modelo ({es(res.loc['Regresión logística', 'ROC-AUC'], 3)}; IC 95 % {es(ic.loc['ROC-AUC'].iloc[1], 3)} – {es(ic.loc['ROC-AUC'].iloc[2], 3)}) "
       f"supera claramente al Dummy (0,5) y alcanza el techo sin fuga ({es(techo['ROC-AUC'], 3)}).")

,Accuracy,Recall,F1,ROC-AUC
"umbral 0,5",0.7248,0.5885,0.7102,0.7746
umbral Youden OOF = 0.806,0.6663,0.4545,0.6095,0.7746
Dummy (prior),0.5730,1.0000,0.7286,0.5000


Al pasar del umbral 0,5 al de Youden, accuracy cambia -0,059, recall -0,134 y F1 -0,101, mientras que el ROC-AUC cambia **0,000**. El Dummy obtiene F1 = 0,729 (mayor que la regresión logística) solo por predecir siempre «tardía»: otra razón para no usar F1 como métrica principal. Con la ROC-AUC, el modelo (0,775; IC 95 % 0,767 – 0,782) supera claramente al Dummy (0,5) y alcanza el techo sin fuga (0,775).

## Respuestas a las preguntas del enunciado

In [4]:
preg = [
    ("¿Qué se predice y para qué?", "Si una línea de pedido llegará tarde (Late_delivery_risk = 1), al crear el pedido, para priorizar acciones antes del despacho."),
    ("¿Por qué ROC-AUC?", "Mide la calidad del orden por riesgo (el uso previsto), no depende del umbral y las clases están casi balanceadas (≈ 57/43). Ver «Métrica elegida»."),
    ("¿Hay fuga?", "Se excluyeron días reales, estado de entrega, estado del pedido y fecha de despacho (cap. 9). Partición y CV temporales con gap; bootstrap por pedido."),
    ("¿Qué estructura tienen los datos?", "Líneas anidadas en pedidos y clientes; marca temporal (eventos, sin estacionalidad en la tasa) y coordenadas del cliente (sin autocorrelación espacial del objetivo)."),
    ("Predictoras más relevantes", f"shipping_mode (AUC univariado {es(rank.iloc[0]['AUC univariado'], 3)}) y, solo en Same Day, la jornada del pedido (corte del mediodía). "
     "El resto tiene AUC ≈ 0,50; payment_type solo parecía útil por los pedidos cancelados/fraude."),
    ("¿Autocorrelación espacial del objetivo?", f"No: I de Moran = {es(moran.iloc[0]['I'], 3)}, p = {es(moran.iloc[0]['p_permutacion'], 2)}."),
    ("¿Generaliza en el tiempo?", "Sí: el ROC-AUC es igual antes y después del cambio de régimen de oct-2017; las variables del modelo no derivan."),
    ("¿Es justo entre grupos?", "ROC-AUC por mercado y segmento entre " + es(sub[sub.variable.isin(['market', 'customer_segment'])]['ROC-AUC'].min(), 3) + " y " + es(sub[sub.variable.isin(['market', 'customer_segment'])]['ROC-AUC'].max(), 3) + "."),
]
pd.DataFrame(preg, columns=["pregunta", "respuesta"])

,pregunta,respuesta
0,¿Qué se predice y para qué?,"Si una línea de pedido llegará tarde (Late_delivery_risk = 1), al ..."
1,¿Por qué ROC-AUC?,"Mide la calidad del orden por riesgo (el uso previsto), no depende..."
2,¿Hay fuga?,"Se excluyeron días reales, estado de entrega, estado del pedido y ..."
3,¿Qué estructura tienen los datos?,"Líneas anidadas en pedidos y clientes; marca temporal (eventos, si..."
4,Predictoras más relevantes,"shipping_mode (AUC univariado 0,727) y, solo en Same Day, la jorna..."
5,¿Autocorrelación espacial del objetivo?,"No: I de Moran = 0,015, p = 0,57."
6,¿Generaliza en el tiempo?,Sí: el ROC-AUC es igual antes y después del cambio de régimen de o...
7,¿Es justo entre grupos?,"ROC-AUC por mercado y segmento entre 0,765 y 0,780."


## Hallazgos principales del EDA que guiaron el modelado

1. **La etiqueta se genera con reglas simples** (cap. 9): Standard y Second Class tienen días
   reales uniformes entre 2 y 6 (retraso aleatorio de 40 % y 80 %), First Class siempre llega
   tarde y Same Day llega tarde si se pide después del mediodía. Esto fija el techo sin fuga.
2. **La hora importa solo en Same Day** (cap. 7): una hora de corte; se modela con la
   interacción modo × jornada.
3. **Pedidos cancelados/fraude** tienen etiqueta 0 por construcción y convierten al medio de
   pago en un proxy (cap. 9): se excluyen de la población.
4. **Los mercados de destino rotan en el tiempo** (cap. 7–8): el destino está confundido con el
   periodo y no aporta; queda fuera del modelo, lo que lo protege de la deriva de la prueba.
5. **Sin estructura temporal ni espacial en el objetivo** (cap. 7–8): residuos sin ACF ni I de
   Moran (cap. 11).
6. **Redundancias**: seis columnas duplicadas, días programados ≡ modo de envío, y
   variables de ventas/ganancia aritméticamente ligadas (cap. 2 y 5).

## Limitaciones

In [5]:
md_out("- **Datos simulados:** DataCo es una empresa ficticia; las reglas que generan la etiqueta son simples y "
       "parte del resultado (retrasos uniformes en Standard y Second Class) es aleatoria por diseño. Las conclusiones no se extrapolan a una cadena real.\n"
       "- **Población:** solo pedidos despachados (sin cancelados ni fraude); clientes en EE. UU. y Puerto Rico; un catálogo deportivo.\n"
       "- **Geografía:** las coordenadas son del cliente/tienda (una por cliente) y algunas tienen errores de signo; el destino solo tiene país/región. "
       "Hay fuerte sesgo de cobertura (áreas metropolitanas).\n"
       "- **Tiempo:** zona horaria desconocida; la llegada de pedidos es casi determinística (artefacto de simulación); "
       "cambio de régimen en oct-2017 (pedidos de una línea).\n"
       "- **Modelo:** lineal y con una sola variable categórica; la separación casi completa hace que la magnitud de los coeficientes dependa de la regularización.\n"
       f"- **Incertidumbre:** los intervalos remuestrean pedidos, no clientes ni días; el ancho del IC del ROC-AUC es ≈ {es(ic.loc['ROC-AUC'].iloc[2] - ic.loc['ROC-AUC'].iloc[1], 3)}.")

- **Datos simulados:** DataCo es una empresa ficticia; las reglas que generan la etiqueta son simples y parte del resultado (retrasos uniformes en Standard y Second Class) es aleatoria por diseño. Las conclusiones no se extrapolan a una cadena real.
- **Población:** solo pedidos despachados (sin cancelados ni fraude); clientes en EE. UU. y Puerto Rico; un catálogo deportivo.
- **Geografía:** las coordenadas son del cliente/tienda (una por cliente) y algunas tienen errores de signo; el destino solo tiene país/región. Hay fuerte sesgo de cobertura (áreas metropolitanas).
- **Tiempo:** zona horaria desconocida; la llegada de pedidos es casi determinística (artefacto de simulación); cambio de régimen en oct-2017 (pedidos de una línea).
- **Modelo:** lineal y con una sola variable categórica; la separación casi completa hace que la magnitud de los coeficientes dependa de la regularización.
- **Incertidumbre:** los intervalos remuestrean pedidos, no clientes ni días; el ancho del IC del ROC-AUC es ≈ 0,016.

## Usos apropiados y consideraciones éticas

El archivo incluye nombres, direcciones y códigos postales de clientes; se excluyeron del
análisis y del modelo, y el análisis espacial se agregó en celdas de 1°. El modelo solo usa
modo de envío y jornada del pedido, de modo que no discrimina por cliente, segmento o
mercado (desempeño homogéneo, cap. 11). Su uso apropiado es **operativo y agregado**:
detectar que la promesa de First Class y el corte de Same Day generan retrasos sistemáticos,
más que etiquetar clientes. Es un trabajo académico, no validado en una empresa real.

## Próximos pasos (entregables siguientes)

- Modelos no lineales (árboles, *boosting*) para confirmar que no hay señal adicional sin fuga
  (se espera que no la haya: el techo es estructural).
- Reformular el objetivo hacia algo con más señal y valor operativo: por ejemplo, predecir los
  **días reales** de envío (regresión) o el retraso en pedidos **Standard**, sabiendo que en la
  simulación es aleatorio.
- Analizar costos de falsas alarmas vs retrasos no anticipados para fijar el umbral.

## Checklist del enunciado

In [6]:
checklist = [
    ("Definición del problema, justificación, fuente, licencia", "Cap. 1"),
    ("Diccionario de variables (tipo, unidad, significado)", "Cap. 2 · tables/diccionario_datos.csv"),
    ("Estructura: unidad, agregación, tipo (espacio-temporal)", "Cap. 2"),
    ("Tamaño ≥ 20.000, n/p, casos por clase, entidades independientes", "Cap. 1–3"),
    ("Faltantes: patrón y mecanismo", "Cap. 2"),
    ("Duplicados y casi-duplicados, outliers, valores imposibles", "Cap. 2, 4 y 6"),
    ("Sesgos de muestreo y ética", "Cap. 2, 8 y 12"),
    ("Reserva del conjunto de prueba antes de decidir", "Cap. 1 (partición cronológica con gap)"),
    ("2.1 Variable objetivo", "Cap. 3"),
    ("2.2 Univariado", "Cap. 4"),
    ("2.3 Bivariado (tamaños de efecto, FDR, VIF, información mutua)", "Cap. 5"),
    ("2.4 Multivariado (PCA, t-SNE, Mahalanobis, Isolation Forest, clustering)", "Cap. 6"),
    ("2.5 Auditoría de fuga", "Cap. 9"),
    ("2.6 Componente temporal (STL, ADF/KPSS, ACF/PACF, deriva)", "Cap. 7"),
    ("2.7 Componente espacial (CRS, mapas, Ripley, DBSCAN, Moran, LISA, Gi*, MAUP)", "Cap. 8"),
    ("2.8 Espacio-temporal (mapas por periodo, Knox, persistencia de hotspots)", "Cap. 8"),
    ("2.9 Preprocesamiento en Pipeline ajustado solo con train", "Cap. 10"),
    ("Línea base trivial (DummyClassifier) y regresión logística", "Cap. 11"),
    ("Validación cruzada del mismo tipo que la partición", "Cap. 11 (temporal con gap; espacial por separado)"),
    ("Métricas: accuracy, recall, F1, ROC-AUC (principal), AP, Brier, matriz, ROC, PR, calibración", "Cap. 11 y 12"),
    ("Intervalos de confianza (bootstrap)", "Cap. 11 (por pedido)"),
    ("Residuos: ACF temporal e I de Moran espacial", "Cap. 11"),
    ("Curva de aprendizaje", "Cap. 11"),
    ("Interpretación de coeficientes y métricas", "Cap. 11"),
    ("Nota crítica (accuracy ≥ 80–90 %)", "Cap. 11, sección 11"),
    ("Jupyter Book, datos, requirements.txt, semillas", "Este repositorio · semilla 42"),
]
cl = pd.DataFrame(checklist, columns=["requisito", "dónde"])
cl.to_csv(TABLES / "checklist_entregable.csv", index=False)
cl

,requisito,dónde
0,"Definición del problema, justificación, fuente, licencia",Cap. 1
1,"Diccionario de variables (tipo, unidad, significado)",Cap. 2 · tables/diccionario_datos.csv
2,"Estructura: unidad, agregación, tipo (espacio-temporal)",Cap. 2
3,"Tamaño ≥ 20.000, n/p, casos por clase, entidades independientes",Cap. 1–3
4,Faltantes: patrón y mecanismo,Cap. 2
5,"Duplicados y casi-duplicados, outliers, valores imposibles","Cap. 2, 4 y 6"
6,Sesgos de muestreo y ética,"Cap. 2, 8 y 12"
7,Reserva del conjunto de prueba antes de decidir,Cap. 1 (partición cronológica con gap)
8,2.1 Variable objetivo,Cap. 3
9,2.2 Univariado,Cap. 4


## Reproducibilidad: semillas y versiones

In [7]:
import sklearn, scipy, matplotlib, statsmodels
versiones = pd.DataFrame({"paquete": ["Python", "numpy", "pandas", "scipy", "scikit-learn", "statsmodels", "matplotlib", "seaborn"],
                          "versión": [platform.python_version(), np.__version__, pd.__version__, scipy.__version__, sklearn.__version__,
                                      statsmodels.__version__, matplotlib.__version__, sns.__version__]})
md_out(f"Semilla global: **{RANDOM_STATE}** (partición, pliegues, bootstrap, t-SNE, k-means, Isolation Forest, permutaciones). "
       f"Partición guardada en `tables/particion_pedidos.csv`.")
versiones

Semilla global: **42** (partición, pliegues, bootstrap, t-SNE, k-means, Isolation Forest, permutaciones). Partición guardada en `tables/particion_pedidos.csv`.

,paquete,versión
0,Python,3.11.15
1,numpy,2.2.6
2,pandas,2.2.3
3,scipy,1.15.3
4,scikit-learn,1.6.1
5,statsmodels,0.14.4
6,matplotlib,3.10.3
7,seaborn,0.13.2


## Referencias

```{bibliography}
:style: plain
```